# sim-app tutorial

This notebook demonstrates the app-facing `albis` workflow:

1. generate one synthetic spatial-transcriptomics dataset,
2. inspect the returned `AnnData`,
3. plot aligned and unaligned coordinates,
4. save the result as `.h5ad`.

It uses small settings so the tutorial stays interactive.

## Setup

From the repository root, create the tutorial environment before opening this notebook:

```bash
bash env/create_tutorial_env.sh
conda activate sim-app-tutorial
jupyter-notebook --no-browser --ip=0.0.0.0 --port 8888
```

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "albis" / "__init__.py").is_file():
            return candidate
        nested = candidate / "sim_app"
        if (nested / "pyproject.toml").is_file() and (nested / "albis" / "__init__.py").is_file():
            return nested
    raise RuntimeError("Could not find the albis repository root.")

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

sys.modules.pop("albis", None)
import albis as ab

print("Python:", sys.executable)
print("albis module:", getattr(ab, "__file__", "<no __file__>"))
print("albis version:", getattr(ab, "__version__", "<no __version__>"))
print("has generate_data:", hasattr(ab, "generate_data"))
print("public names:", [name for name in dir(ab) if not name.startswith("_")][:30])

## Generate a small bin-level dataset

`generate_data()` returns exactly one `AnnData` object for the requested output type and slice axis.

In [ ]:
adata = ab.generate_data(
    output="bin",
    slice_axis="Z",
    n_cells=10000,
    n_slices=4,
    n_domains=5,
    marker_genes_per_type=20,
    sphere_radius_um=200.0,
    capture_window_um=(300.0, 300.0),
    bin_size_um=30.0,
    seed=2025,
)

adata

## Inspect the returned object

`describe()` gives a compact summary of the app-relevant fields.

In [ ]:
summary = ab.describe(adata)
summary

In [ ]:
adata.obs.head()

In [ ]:
adata.var.head()

## Plot aligned and unaligned coordinates

`plot()` returns a Matplotlib figure. Use `coordinates="aligned"` for canonical tissue coordinates and `coordinates="unaligned"` for the per-slice rigid transforms.

In [ ]:
fig = ab.plot(
    adata,
    view="2d",
    coordinates="aligned",
    color="domain_true",
    point_size=8,
)
fig

In [ ]:
fig = ab.plot(
    adata,
    view="2d",
    coordinates="unaligned",
    color="slice_id",
    point_size=8,
)
fig

## Generate other app outputs

Use the same public API for spot-level or cell-level data. Only the requested output is generated.

In [ ]:
spot_adata = ab.example_data(output="spot", slice_axis="Y", n_cells=200)
ab.describe(spot_adata)

In [ ]:
cell_adata = ab.example_data(output="cell", slice_axis="X", n_cells=200)
ab.describe(cell_adata)

## Watch out: `domain_type_mix` and non-default domain/cell-type counts

`generate_data()` lets you change `n_domains`/`n_cell_types` freely, but if
you do and don't also supply `domain_type_mix`, every domain silently gets
an identical, fully uniform cell-type distribution -- no domain-specific
enrichment at all. This raises a `UserWarning` so it's impossible to miss:


In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _ = ab.generate_data(output="cell", n_cells=200, n_domains=3, n_cell_types=5)
    domain_mix_warnings = [str(w.message) for w in caught if "domain_type_mix" in str(w.message)]

print(domain_mix_warnings[0] if domain_mix_warnings else "(no warning raised)")


Fix it by supplying your own `(n_domains, n_cell_types)` matrix -- each row
gets renormalized to a probability distribution automatically, so it doesn't
need to sum to exactly 1:


In [ ]:
import numpy as np

my_mix = np.array([
    [0.6, 0.2, 0.1, 0.05, 0.05],   # domain 0: enriched for type 0
    [0.1, 0.6, 0.2, 0.05, 0.05],   # domain 1: enriched for type 1
    [0.2, 0.2, 0.2, 0.2, 0.2],     # domain 2 (core): uniform, on purpose
])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    adata_custom_mix = ab.generate_data(
        output="cell", n_cells=200, n_domains=3, n_cell_types=5, domain_type_mix=my_mix,
    )
    print("warnings raised:", len(caught))


## Save the result

`save()` writes the object with AnnData's `.h5ad` format and returns the output path.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path = ab.save(adata, output_dir / "simulation_bins_z.h5ad")
output_path